# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [35]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import roc_auc_score, average_precision_score
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
df.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?

In [3]:
# Target variable distribution

target_counts = df["fraud"].value_counts().sort_index()
target_percentages = df["fraud"].value_counts(normalize=True).sort_index() * 100

target_distribution = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages.round(2)
})

target_distribution

,count,percentage
fraud,,
0.0,912597,91.26
1.0,87403,8.74


### Target Variable Distribution

The target variable `fraud` is clearly imbalanced.

Most transactions are legitimate, representing around 91% of the dataset, while fraudulent transactions represent only around 9%.

This means we are dealing with an imbalanced classification problem. Because of this, accuracy alone will not be a reliable metric to evaluate the model. A model could achieve high accuracy by mostly predicting the majority class, while still performing poorly at detecting fraud.

For this reason, in the next steps we should pay special attention to metrics such as recall, precision, F1-score, and the confusion matrix, especially for the fraudulent class.

- **2.** Train a LogisticRegression.

In [4]:
X = df.drop(columns="fraud")
y = df["fraud"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y)

In [6]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (800000, 7)
X_test shape: (200000, 7)
y_train shape: (800000,)
y_test shape: (200000,)


In [8]:
# Logistic Regression pipeline

log_reg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("log_reg", LogisticRegression(max_iter=1000, random_state=42))
])

In [9]:
# Train the model

log_reg_model.fit(X_train, y_train)

,steps,"[('scaler', ...), ('log_reg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


In [10]:
# Predictions

y_pred = log_reg_model.predict(X_test)

In [11]:
# Predicted probabilities for the positive class

y_pred_proba = log_reg_model.predict_proba(X_test)[:, 1]

### Logistic Regression Model

We trained a Logistic Regression model as our baseline classifier.

Before training the model, we separated the dataset into features `X` and target variable `y`. Then, we applied a train-test split using stratification to preserve the original class distribution of the target variable in both sets.

Since Logistic Regression is sensitive to the scale of the input variables, we used a pipeline including `StandardScaler` before fitting the model.

This model will be used as the baseline before applying balancing techniques such as oversampling, undersampling, and SMOTE.

- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.

In [14]:
# Model evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba)

log_reg_results = pd.DataFrame({
    "model": ["Logistic Regression"],
    "accuracy": [accuracy],
    "precision": [precision],
    "recall": [recall],
    "f1_score": [f1],
    "roc_auc": [roc_auc],
    "pr_auc": [pr_auc]
})

log_reg_results

,model,accuracy,precision,recall,f1_score,roc_auc,pr_auc
0,Logistic Regression,0.95941,0.896435,0.605572,0.722841,0.966977,0.807224


In [15]:
print(classification_report(y_test, y_pred, target_names=["Legit", "Fraud"]))

              precision    recall  f1-score   support

       Legit       0.96      0.99      0.98    182519
       Fraud       0.90      0.61      0.72     17481

    accuracy                           0.96    200000
   macro avg       0.93      0.80      0.85    200000
weighted avg       0.96      0.96      0.96    200000



In [16]:
cm = confusion_matrix(y_test, y_pred)

cm

array([[181296,   1223],
       [  6895,  10586]])

### Baseline Logistic Regression Results

The baseline Logistic Regression model achieved a high accuracy of 95.94%. However, since the dataset is imbalanced, accuracy can be misleading.

The model achieved a precision of 89.64%, meaning that when it predicts fraud, it is usually correct. However, the recall is only 60.56%, meaning that the model detects only around 60% of all actual fraudulent transactions.

The confusion matrix shows that the model correctly detected 10,586 fraud cases, but it missed 6,895 fraudulent transactions, classifying them as legitimate.

In a fraud detection problem, false negatives are especially dangerous because they represent fraud cases that go undetected. Therefore, despite the high accuracy, the baseline model is not optimal for this business problem.

The model is conservative: it avoids falsely accusing legitimate transactions, but at the cost of missing many real fraud cases.

For this reason, we will now try balancing techniques such as oversampling, undersampling, and SMOTE to see whether the recall for the fraud class improves.

- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 

In [18]:
# Oversampling

oversampler = RandomOverSampler(random_state=42)

X_train_over, y_train_over = oversampler.fit_resample(X_train, y_train)

In [19]:
print("Original training target distribution:")
print(y_train.value_counts())

print("\nOversampled training target distribution:")
print(y_train_over.value_counts())

Original training target distribution:
fraud
0.0    730078
1.0     69922
Name: count, dtype: int64

Oversampled training target distribution:
fraud
0.0    730078
1.0    730078
Name: count, dtype: int64


In [20]:
# Logistic Regression with oversampled training data

log_reg_over = Pipeline([
    ("scaler", StandardScaler()),
    ("log_reg", LogisticRegression(max_iter=1000, random_state=42))
])

log_reg_over.fit(X_train_over, y_train_over)

,steps,"[('scaler', ...), ('log_reg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


In [21]:
# Predictions on the original test set

y_pred_over = log_reg_over.predict(X_test)
y_pred_proba_over = log_reg_over.predict_proba(X_test)[:, 1]

In [22]:
# Evaluation metrics for oversampled model

accuracy_over = accuracy_score(y_test, y_pred_over)
precision_over = precision_score(y_test, y_pred_over)
recall_over = recall_score(y_test, y_pred_over)
f1_over = f1_score(y_test, y_pred_over)
roc_auc_over = roc_auc_score(y_test, y_pred_proba_over)
pr_auc_over = average_precision_score(y_test, y_pred_proba_over)

log_reg_over_results = pd.DataFrame({
    "model": ["Logistic Regression Oversampling"],
    "accuracy": [accuracy_over],
    "precision": [precision_over],
    "recall": [recall_over],
    "f1_score": [f1_over],
    "roc_auc": [roc_auc_over],
    "pr_auc": [pr_auc_over]
})

log_reg_over_results

,model,accuracy,precision,recall,f1_score,roc_auc,pr_auc
0,Logistic Regression Oversampling,0.934785,0.577322,0.947772,0.717556,0.97953,0.757315


In [23]:
comparison_results = pd.concat([log_reg_results, log_reg_over_results], ignore_index=True)

comparison_results

,model,accuracy,precision,recall,f1_score,roc_auc,pr_auc
0,Logistic Regression,0.959410,0.896435,0.605572,0.722841,0.966977,0.807224
1,Logistic Regression Oversampling,0.934785,0.577322,0.947772,0.717556,0.979530,0.757315


### Oversampling Results

After applying Random Oversampling, the Logistic Regression model became much more sensitive to the fraudulent class.

The recall increased from 60.56% to 94.78%, meaning that the model is now able to detect almost all fraudulent transactions. This is a major improvement in terms of fraud detection, since false negatives are especially dangerous in this business context.

However, this improvement comes at a clear cost. Precision decreased from 89.64% to 57.73%, meaning that the model now produces many more false positives. In other words, many legitimate transactions are incorrectly classified as fraud.

The F1-score slightly decreased from 0.7228 to 0.7176, and the PR-AUC also decreased from 0.8072 to 0.7573. Therefore, although oversampling improves recall significantly, it does not clearly improve the overall performance of the model.

In conclusion, oversampling is useful if the main business priority is to detect as many fraud cases as possible. However, if we also care about reducing false alarms, the baseline model remains more balanced.

- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?

In [26]:
# Undersampling

undersampler = RandomUnderSampler(random_state=42)

X_train_under, y_train_under = undersampler.fit_resample(X_train, y_train)

In [27]:
print("Original training target distribution:")
print(y_train.value_counts())

print("\nUndersampled training target distribution:")
print(y_train_under.value_counts())

Original training target distribution:
fraud
0.0    730078
1.0     69922
Name: count, dtype: int64

Undersampled training target distribution:
fraud
0.0    69922
1.0    69922
Name: count, dtype: int64


In [28]:
undersampled_distribution = pd.DataFrame({
    "count": y_train_under.value_counts().sort_index(),
    "percentage": (y_train_under.value_counts(normalize=True).sort_index() * 100).round(2)
})

undersampled_distribution

,count,percentage
fraud,,
0.0,69922,50.0
1.0,69922,50.0


In [29]:
# Logistic Regression with undersampled training data

log_reg_under = Pipeline([
    ("scaler", StandardScaler()),
    ("log_reg", LogisticRegression(max_iter=1000, random_state=42))
])

log_reg_under.fit(X_train_under, y_train_under)

,steps,"[('scaler', ...), ('log_reg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


In [30]:
# Predictions on the original test set

y_pred_under = log_reg_under.predict(X_test)
y_pred_proba_under = log_reg_under.predict_proba(X_test)[:, 1]

In [31]:
# Evaluation metrics for undersampled model

accuracy_under = accuracy_score(y_test, y_pred_under)
precision_under = precision_score(y_test, y_pred_under)
recall_under = recall_score(y_test, y_pred_under)
f1_under = f1_score(y_test, y_pred_under)
roc_auc_under = roc_auc_score(y_test, y_pred_proba_under)
pr_auc_under = average_precision_score(y_test, y_pred_proba_under)

log_reg_under_results = pd.DataFrame({
    "model": ["Logistic Regression Undersampling"],
    "accuracy": [accuracy_under],
    "precision": [precision_under],
    "recall": [recall_under],
    "f1_score": [f1_under],
    "roc_auc": [roc_auc_under],
    "pr_auc": [pr_auc_under]
})

log_reg_under_results

,model,accuracy,precision,recall,f1_score,roc_auc,pr_auc
0,Logistic Regression Undersampling,0.93477,0.577289,0.947486,0.717448,0.979561,0.756823


In [32]:
comparison_results = pd.concat(
    [log_reg_results, log_reg_over_results, log_reg_under_results],
    ignore_index=True
)

comparison_results

,model,accuracy,precision,recall,f1_score,roc_auc,pr_auc
0,Logistic Regression,0.959410,0.896435,0.605572,0.722841,0.966977,0.807224
1,Logistic Regression Oversampling,0.934785,0.577322,0.947772,0.717556,0.979530,0.757315
2,Logistic Regression Undersampling,0.934770,0.577289,0.947486,0.717448,0.979561,0.756823


In [33]:
cm_under = confusion_matrix(y_test, y_pred_under)

cm_under

array([[170391,  12128],
       [   918,  16563]])

### Undersampling Results

After applying Random Undersampling, the Logistic Regression model achieved a recall of 94.75%, which is much higher than the baseline model's recall of 60.56%.

This means the model became much better at detecting fraudulent transactions. The number of false negatives decreased from 6,895 in the baseline model to only 918 after undersampling.

However, this improvement came at the cost of a much lower precision. Precision decreased from 89.64% to 57.73%, meaning that the model now produces many more false positives.

The number of legitimate transactions incorrectly classified as fraud increased from 1,223 to 12,128.

Compared with oversampling, undersampling produced almost identical results. Both techniques strongly improved recall but reduced precision and slightly decreased the F1-score and PR-AUC.

In conclusion, undersampling improves fraud detection, but it does not clearly improve the overall model performance. It also has the disadvantage of removing many legitimate transactions from the training data.

- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

In [36]:
# SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [37]:
print("Original training target distribution:")
print(y_train.value_counts())

print("\nSMOTE training target distribution:")
print(y_train_smote.value_counts())

Original training target distribution:
fraud
0.0    730078
1.0     69922
Name: count, dtype: int64

SMOTE training target distribution:
fraud
0.0    730078
1.0    730078
Name: count, dtype: int64


In [38]:
# Logistic Regression with SMOTE

log_reg_smote = Pipeline([
    ("scaler", StandardScaler()),
    ("log_reg", LogisticRegression(max_iter=1000, random_state=42))
])

log_reg_smote.fit(X_train_smote, y_train_smote)

,steps,"[('scaler', ...), ('log_reg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


In [39]:
# Predictions on the original test set

y_pred_smote = log_reg_smote.predict(X_test)
y_pred_proba_smote = log_reg_smote.predict_proba(X_test)[:, 1]

In [40]:
# Evaluation metrics for SMOTE model

accuracy_smote = accuracy_score(y_test, y_pred_smote)
precision_smote = precision_score(y_test, y_pred_smote)
recall_smote = recall_score(y_test, y_pred_smote)
f1_smote = f1_score(y_test, y_pred_smote)
roc_auc_smote = roc_auc_score(y_test, y_pred_proba_smote)
pr_auc_smote = average_precision_score(y_test, y_pred_proba_smote)

log_reg_smote_results = pd.DataFrame({
    "model": ["Logistic Regression SMOTE"],
    "accuracy": [accuracy_smote],
    "precision": [precision_smote],
    "recall": [recall_smote],
    "f1_score": [f1_smote],
    "roc_auc": [roc_auc_smote],
    "pr_auc": [pr_auc_smote]
})

log_reg_smote_results

,model,accuracy,precision,recall,f1_score,roc_auc,pr_auc
0,Logistic Regression SMOTE,0.935175,0.579045,0.946227,0.71844,0.979175,0.761676


In [41]:
comparison_results = pd.concat(
    [log_reg_results, log_reg_over_results, log_reg_under_results, log_reg_smote_results],
    ignore_index=True
)

comparison_results

,model,accuracy,precision,recall,f1_score,roc_auc,pr_auc
0,Logistic Regression,0.959410,0.896435,0.605572,0.722841,0.966977,0.807224
1,Logistic Regression Oversampling,0.934785,0.577322,0.947772,0.717556,0.979530,0.757315
2,Logistic Regression Undersampling,0.934770,0.577289,0.947486,0.717448,0.979561,0.756823
3,Logistic Regression SMOTE,0.935175,0.579045,0.946227,0.718440,0.979175,0.761676


In [42]:
cm_smote = confusion_matrix(y_test, y_pred_smote)

cm_smote

array([[170494,  12025],
       [   940,  16541]])

## Final Conclusions

In this notebook, we worked with a credit card fraud detection dataset where the target variable was `fraud`.

The target variable was clearly imbalanced. Legitimate transactions represented around 91% of the dataset, while fraudulent transactions represented only around 9%. Because of this imbalance, accuracy alone was not enough to evaluate the model properly.

For this problem, the most important class is the fraudulent class, represented by `1`. In a fraud detection context, false negatives are especially dangerous because they represent fraudulent transactions that the model classifies as legitimate. Therefore, recall, precision, F1-score, PR-AUC, and the confusion matrix were more relevant than accuracy alone.

We first trained a baseline Logistic Regression model using the original imbalanced training data. Then, we applied three balancing techniques only to the training set:

- Random Oversampling
- Random Undersampling
- SMOTE

The test set was kept unchanged in all cases in order to evaluate the models on the original real-world distribution.

### Model Comparison

| Model | Accuracy | Precision | Recall | F1-score | ROC-AUC | PR-AUC |
|---|---:|---:|---:|---:|---:|---:|
| Logistic Regression | 0.9594 | 0.8964 | 0.6056 | 0.7228 | 0.9670 | 0.8072 |
| Logistic Regression Oversampling | 0.9348 | 0.5773 | 0.9478 | 0.7176 | 0.9795 | 0.7573 |
| Logistic Regression Undersampling | 0.9348 | 0.5773 | 0.9475 | 0.7174 | 0.9796 | 0.7568 |
| Logistic Regression SMOTE | 0.9352 | 0.5790 | 0.9462 | 0.7184 | 0.9792 | 0.7617 |

### Interpretation

The baseline Logistic Regression model achieved the highest accuracy, precision, F1-score, and PR-AUC. However, its recall was only 60.56%, meaning that it failed to detect a large number of fraudulent transactions.

The baseline confusion matrix showed that the model missed 6,895 fraud cases, classifying them as legitimate. This is a serious limitation in a fraud detection problem.

After applying balancing techniques, recall improved dramatically:

- Baseline recall: 60.56%
- Oversampling recall: 94.78%
- Undersampling recall: 94.75%
- SMOTE recall: 94.62%

This means that the balanced models detected many more fraudulent transactions.

However, this improvement came with a clear trade-off. Precision dropped from 89.64% in the baseline model to around 58% in the balanced models. This means that the balanced models produced many more false positives, incorrectly classifying legitimate transactions as fraud.

For example, with SMOTE, the model reduced false negatives from 6,895 to 940, which is a major improvement. However, false positives increased from 1,223 to 12,025.

### Does balancing improve the model?

Balancing improves the model if the main goal is to detect as many fraud cases as possible. All three balancing techniques significantly improved recall and reduced false negatives.

However, balancing does not clearly improve the overall model performance. The baseline model still achieved the best F1-score and PR-AUC, which means it had a better balance between precision and recall.

Among the balancing techniques, SMOTE produced the best F1-score and PR-AUC, although the differences compared with oversampling and undersampling were small. Oversampling achieved the highest recall, but SMOTE offered a slightly better balance overall.

### Final Decision

If the business priority is to avoid missing fraud cases, a balanced model such as SMOTE or Oversampling would be preferred because it detects far more fraudulent transactions.

If the business priority is to reduce false alarms and maintain a stronger balance between precision and recall, the baseline Logistic Regression model remains the strongest option.

In this specific fraud detection context, recall is highly important because missing fraud can be very costly. Therefore, the balanced models are useful, especially SMOTE, but they should be used with awareness of the increase in false positives.

A good next step would be to adjust the classification threshold instead of relying only on the default threshold of 0.5. This could help find a better balance between detecting fraud and reducing false alarms.